# Water Quality Data Cleaning: EPA Stations

In [1]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import folium
from shapely.geometry import Point
from pathlib import Path

In [ ]:
data_path = r'../../../../data/tabular/water-quality/raw'
df_station = pd.read_csv(f'{data_path}/epa-stations.csv')

In [ ]:
# Step 1: Inspect
print(df_station.shape)
df_station.head()

In [ ]:
# Step 2: Rename columns - strip spaces, replace with underscores
df_station.columns = df_station.columns.str.strip().str.replace(" ", "_")

# Step 3: Drop columns with >90% missing values
df_station.replace(r'^\s*$', pd.NA, regex=True, inplace=True)
threshold = len(df_station) * 0.10
df_station.dropna(axis=1, thresh=threshold, inplace=True)

# Step 4: Select key columns (only keep ones that survived the drop)
keep_cols = [
    "OrganizationIdentifier",
    "MonitoringLocationIdentifier",
    "MonitoringLocationName",
    "MonitoringLocationTypeName",
    "HUCEightDigitCode",
    "LatitudeMeasure",
    "LongitudeMeasure",
    "DrainageAreaMeasure/MeasureValue",
    "DrainageAreaMeasure/MeasureUnitCode",
    "StateCode",
    "CountyCode",
    "ProviderName"
]
keep_cols = [c for c in keep_cols if c in df_station.columns]
df_station = df_station[keep_cols]

# Step 5: Fix data types
df_station["LatitudeMeasure"] = pd.to_numeric(df_station["LatitudeMeasure"], errors="coerce")
df_station["LongitudeMeasure"] = pd.to_numeric(df_station["LongitudeMeasure"], errors="coerce")
if "DrainageAreaMeasure/MeasureValue" in df_station.columns:
    df_station["DrainageAreaMeasure/MeasureValue"] = pd.to_numeric(
        df_station["DrainageAreaMeasure/MeasureValue"], errors="coerce"
    )

print(f"Shape: {df_station.shape}")
print(f"Columns: {df_station.columns.tolist()}")
df_station.dtypes

In [ ]:
# Step 6: Save Cleaned Data
import os
out_path = '../../../../data/tabular/water-quality/cleaned'
os.makedirs(out_path, exist_ok=True)
df_station.to_csv(f'{out_path}/epa-stations-clean.csv', index=False)
print(f"Saved {len(df_station):,} rows → {out_path}/epa-stations-clean.csv")